<div style="display: flex; background-color: #5B6C9F;" >
<h1 style="margin: auto; padding: 30px; color: white; ">Detecting counterfeit banknotes - Notebook 2 - Applying the model</h1>
</div>

In [1]:
# Import libraries
import pandas as pd
import joblib
import numpy as np

In [2]:
# Load the trained model
modele = joblib.load("../model/counterfeit_detection_model.joblib")

In [3]:
# Load the new banknotes to check
df_nouveaux_billets = pd.read_csv("../data/billets_production.csv", sep=",")

In [4]:
# Size variables used by the model
taille_modele = [
    "diagonal",
    "height_left",
    "height_right",
    "margin_low",
    "margin_up",
    "length",
]

In [5]:
# Keep only the measurements
df_taille_billets = df_nouveaux_billets[taille_modele]

In [6]:
# Predict on the new data
predictions = modele.predict(df_taille_billets)

In [7]:
# Model confidence: probability of being genuine
confiance = modele.predict_proba(df_taille_billets)[:, 1]

In [8]:
# Copy the ids so the original data stays untouched
resultats = df_nouveaux_billets[["id"]].copy()

In [9]:
# Turn the predictions into a readable table
resultats["Prediction"] = ["Genuine" if p else "Counterfeit" for p in predictions]
resultats["Probability of being genuine"] = confiance.round(3)

In [10]:
# Flag uncertain notes for a human check
ecart = np.abs(confiance - 0.5) * 2
seuil = 0.30
resultats["Human check"] = np.where(ecart < seuil, "To check", "")

In [11]:
# Show the results
resultats

,id,Prediction,Probability of being genuine,Human check
0,B_1,Genuine,0.995,
1,B_2,Counterfeit,0.003,
2,B_3,Genuine,0.999,
3,B_4,Counterfeit,0.000,
4,B_5,Counterfeit,0.011,


In [12]:
# Count genuine and counterfeit notes
bilan = resultats["Prediction"].value_counts()
bilan

Prediction
Counterfeit    3
Genuine        2
Name: count, dtype: int64